# GDAL/OGR → STAC Transformer

`dynastore.extensions.stac.metadata_mapper` ships a small **pure transformer** that maps `gdalinfo` / `ogrinfo` JSON output to STAC extension fields. It can be applied to a `pystac.Collection`, `pystac.Item`, or `pystac.Asset`.

## Public surface

```python
from dynastore.extensions.stac.metadata_mapper import (
    StacFields,
    gdal_to_stac_fields,            # pure: dict -> StacFields
    apply_to_collection,            # uses summaries (band schemas at collection scope)
    apply_to_item,                  # bbox + projection properties on Item
    apply_to_asset,                 # raster:bands / eo:bands on Asset
    enrich_collection_from_metadata, # backwards-compat wrapper used by stac_virtual.py
)
```

## What it maps

| Source key (gdalinfo/ogrinfo JSON) | STAC field | Carrier |
| --- | --- | --- |
| `coordinateSystem.wkt` | `proj:wkt2` | extra_fields / properties |
| `size: [w, h]` | `proj:shape: [h, w]` | extra_fields / properties |
| `geoTransform` | `proj:transform` | extra_fields / properties |
| `bands[].type` (GDAL) | `raster:bands[].data_type` | summaries / asset |
| `bands[].noDataValue` | `raster:bands[].nodata` | summaries / asset |
| `bands[].description` | `eo:bands[].description` | summaries / asset |
| `wgs84Extent` / `cornerCoordinates` | bbox | extent.spatial / item.bbox |
| `layers[0].geometryType` | `vector:geometry_type` | extra_fields |
| `layers[0].featureCount` | `vector:count` | extra_fields |
| `layers[0].fields[]` (OGR) | `table:columns[]` | extra_fields |

This notebook is **standalone** — it does not need a running dynastore service.

## 1. Raster example — pure `gdal_to_stac_fields`

Realistic shape of `gdalinfo -json some.tif` (trimmed).

In [ ]:
from dynastore.extensions.stac.metadata_mapper import gdal_to_stac_fields

raster_info = {
    "size": [10980, 10980],
    "coordinateSystem": {"wkt": 'PROJCS["WGS 84 / UTM zone 33N", ...]'},
    "geoTransform": [499980.0, 10.0, 0.0, 5400000.0, 0.0, -10.0],
    "bands": [
        {"band": 1, "type": "UInt16", "description": "Blue",  "noDataValue": 0},
        {"band": 2, "type": "UInt16", "description": "Green", "noDataValue": 0},
        {"band": 3, "type": "UInt16", "description": "Red",   "noDataValue": 0},
        {"band": 4, "type": "UInt16", "description": "NIR",   "noDataValue": 0},
    ],
    "wgs84Extent": {
        "type": "Polygon",
        "coordinates": [[
            [11.0, 48.0], [12.0, 48.0], [12.0, 49.0], [11.0, 49.0], [11.0, 48.0]
        ]],
    },
}

fields = gdal_to_stac_fields(raster_info)
print("Detected:        raster")
print("Extensions:     ", fields.extensions)
print("proj:shape:     ", fields.extra_fields["proj:shape"])
print("proj:transform: ", fields.extra_fields["proj:transform"])
print("bbox:           ", fields.bbox)
print("raster bands:   ", len(fields.raster_bands), "(Blue/Green/Red/NIR)")
print("eo bands:       ", [b.name for b in fields.eo_bands])

## 2. Apply to `pystac.Collection`

Bands attach via STAC **summaries** at collection scope (one Collection = one virtual asset is the shape `stac_virtual.py` uses).

In [ ]:
import datetime as dt
import pystac
from dynastore.extensions.stac.metadata_mapper import apply_to_collection

collection = pystac.Collection(
    id="sentinel2-tile-T33TUN",
    description="Single S2 tile presented as a virtual collection.",
    extent=pystac.Extent(
        spatial=pystac.SpatialExtent([[-180, -90, 180, 90]]),  # placeholder
        temporal=pystac.TemporalExtent([[dt.datetime(2024, 1, 1), None]]),
    ),
    license="proprietary",
)

apply_to_collection(collection, fields)

print("After apply:")
print("  spatial bbox:", collection.extent.spatial.bboxes)
print("  extensions: ", collection.stac_extensions)
print("  summaries:  ", list(collection.summaries.lists.keys()) if collection.summaries else [])

## 3. Apply to `pystac.Item` + `pystac.Asset`

When you have a real Item, `apply_to_item` writes bbox + projection properties onto the Item, and `apply_to_asset` writes `raster:bands` / `eo:bands` onto a specific asset.

**Order matters** — pystac requires the asset to be attached to its owning item *before* the band extensions are applied (the extension URL goes onto the Item, not the Asset).

In [ ]:
from dynastore.extensions.stac.metadata_mapper import apply_to_item, apply_to_asset

item = pystac.Item(
    id="S2A_T33TUN_20240615",
    geometry={"type": "Point", "coordinates": [11.5, 48.5]},  # placeholder
    bbox=[11.0, 48.0, 12.0, 49.0],
    datetime=dt.datetime(2024, 6, 15),
    properties={},
)

asset = pystac.Asset(href="gs://example-bucket/T33TUN/B04.tif", media_type="image/tiff")
item.add_asset("data", asset)            # 1) attach FIRST
apply_to_item(item, fields)               # 2) bbox + proj:* on item
apply_to_asset(item.assets["data"], fields)  # 3) bands on asset

print("Item bbox:        ", item.bbox)
print("Item proj:shape:  ", item.properties.get("proj:shape"))
print("Item extensions: ", item.stac_extensions)
print("Asset extra:     ", list(item.assets["data"].extra_fields.keys()))

## 4. Vector example (OGR)

When the source dict has `layers`, the transformer routes through the vector path: `vector:` extension fields and `table:columns` from OGR field types.

In [ ]:
ogr_info = {
    "layers": [{
        "name": "admin_boundaries",
        "geometryType": "Polygon",
        "featureCount": 16234,
        "coordinateSystem": {"wkt": 'GEOGCS["WGS 84", ...]'},
        "extent": [-180.0, -90.0, 180.0, 83.62],
        "fields": [
            {"name": "adm0_code",  "type": "String"},
            {"name": "adm0_name",  "type": "String"},
            {"name": "area_km2",   "type": "Real"},
            {"name": "created_on", "type": "DateTime"},
        ],
    }]
}

v_fields = gdal_to_stac_fields(ogr_info)
print("Detected:           vector")
print("Extensions:        ", v_fields.extensions)
print("vector:geometry:   ", v_fields.extra_fields["vector:geometry_type"])
print("vector:count:      ", v_fields.extra_fields["vector:count"])
print("bbox:              ", v_fields.bbox)
print("table:columns:     ", v_fields.table_columns)

## 5. Backwards-compat wrapper

`enrich_collection_from_metadata(collection, asset.metadata)` is the **live** entry point used by `stac_virtual.py`. It accepts the full asset.metadata dict — i.e. anything that `GdalInfoTask` writes — and:

1. Copies non-`gdalinfo`, non-`stac*` keys onto `collection.extra_fields["asset:<key>"]`.
2. Runs `gdal_to_stac_fields(metadata['gdalinfo'])` if present.
3. Applies the result to the Collection.

The new pure transformer is what powers it under the hood — old callers don't change.

In [ ]:
from dynastore.extensions.stac.metadata_mapper import enrich_collection_from_metadata

asset_metadata = {
    "gdalinfo": raster_info,
    "publisher": "FAO",
    "acquisition_year": 2024,
    "stac_version": "1.0.0",  # ignored (starts with 'stac')
}

col2 = pystac.Collection(
    id="virtual-asset-T33TUN",
    description="Virtual collection for ingested asset.",
    extent=pystac.Extent(
        spatial=pystac.SpatialExtent([[-180, -90, 180, 90]]),
        temporal=pystac.TemporalExtent([[dt.datetime(2024, 1, 1), None]]),
    ),
    license="proprietary",
)

enrich_collection_from_metadata(col2, asset_metadata)

print("Custom asset fields:")
for k, v in col2.extra_fields.items():
    if k.startswith("asset:"):
        print(f"  {k}: {v}")
print("\nProjection: proj:wkt2 set =", "proj:wkt2" in col2.extra_fields)
print("Skipped 'stac_version' (prefix-stripped)? ",
      "asset:stac_version" not in col2.extra_fields)

## 6. End-to-end with a real file (optional)

If GDAL is available locally, you can drive the whole flow off a real raster/vector by shelling out to `gdalinfo -json`. Skipped automatically when GDAL is not installed.

In [ ]:
import json
import shutil
import subprocess

GDAL_BIN = shutil.which("gdalinfo")
SAMPLE = "/path/to/your.tif"  # <-- set this to a real file to run

if GDAL_BIN is None:
    print("SKIP: gdalinfo not on PATH.")
elif not __import__("os").path.exists(SAMPLE):
    print(f"SKIP: sample file not found at {SAMPLE}; set SAMPLE to a real path to run.")
else:
    raw = subprocess.check_output([GDAL_BIN, "-json", SAMPLE], text=True)
    info = json.loads(raw)
    live_fields = gdal_to_stac_fields(info)
    print("Extensions:", live_fields.extensions)
    print("bbox:      ", live_fields.bbox)
    print("bands:     ", len(live_fields.raster_bands))

## Recap

- `gdal_to_stac_fields(dict) -> StacFields` is **pure** — chainable into any pipeline (asset enrichment, item-level mapping, ingestion-time pre-population). No pystac side effects.
- Three thin applicators (`apply_to_collection`, `apply_to_item`, `apply_to_asset`) cover the standard STAC carriers.
- `enrich_collection_from_metadata` is preserved as the wrapper used by `stac_virtual.py` — existing callers keep working unchanged.

## Where it's wired today

- **Producer**: `tasks/gdal/gdalinfo_task.py` writes `asset.metadata['gdalinfo'] = info`.
- **Consumer**: `extensions/stac/stac_virtual.py` calls `enrich_collection_from_metadata(collection, asset.metadata)` when a virtual collection is requested for an asset.

Lift the pure transformer wherever you need STAC fields derived from GDAL/OGR JSON — for example, populating `proj:`/`raster:`/`eo:` on real Items at ingest time.